In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

project_root = Path.cwd()
while project_root.name != 'python' and project_root.parent != project_root:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from data import get_log_returns;

## Downloading the data
#### Source: _Yahoo Finance_

In [ ]:
from data import download_tickers_history

# set the date range for the historic data
start_date = datetime(year=2020, month=1, day=1)
end_date = datetime(year=2025, month=12, day=31)
history = download_tickers_history(start_date, end_date, ['NVDA']);

nvda = history.NVDA;


## Volatility and volatility clustering
We will be thinking about volatility as the _std_ over the _returns_

Trying to build a _Rolling Volatility_ over a window of, 20 trading days (roughly equivalent to a trading month)

In [ ]:
from data import get_rolling_vol_daily

log_returns = get_log_returns(history)

window = 20
rolling_vol_daily = get_rolling_vol_daily(history, window)

TRADING_DAYS_PER_YEAR = 252
rolling_vol_annual = rolling_vol_daily * np.sqrt(TRADING_DAYS_PER_YEAR) * 100 # (%)

dates = log_returns.index[window:]

# data visualization
plt.figure(figsize=(10, 6))

plt.plot(
    dates,
    rolling_vol_annual,
    linewidth=1,
    label='20-Day Rolling Volatility (Annualized)'
);

plt.title('NVDA: 20-Day Rolling Volatility')
plt.xlabel('Periods (1 period = 1 day)', fontsize=12)
plt.ylabel('Annualized Volatility (%)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

plt.show()

## Volatility x Volume
As a rule, the larger the volume traded in the market, the more volatile the ticket becomes. If the volatility is relatively high with normal or low trading volume over a certain period of time, it means that speculation dominates the market.

In [ ]:
import matplotlib.ticker as ticker

nvda = history.NVDA

log_returns = get_log_returns(history)
abs_log_returns_perc = np.abs(log_returns.NVDA) * 100 # (%)

volume = nvda.Volume.values[1:]

v_min, v_max = np.min(volume), np.max(volume)
size = 15 + (volume - v_min) / (v_max - v_min) * 185

# data visualization
fig, ax = plt.subplots(figsize=(10, 6));

scatter = ax.scatter(volume, abs_log_returns_perc, c=abs_log_returns_perc, s=size, cmap="Spectral_r")

# disable scientific notation & offset on X-axis to show full numbers
ax.ticklabel_format(style='plain', useOffset=False, axis='x')

ax.set_title("NVDA: Volume vs. Volatility (MDH Funnel Effect)", fontsize=14, fontweight='bold')
ax.set_xlabel("Daily Volume (in millions)")
ax.set_ylabel("Absolute Log Returns (%)")

ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'{x/1e6:.0f}M'))
legend1 = ax.legend(*scatter.legend_elements(num=5),
                    loc="upper left", title="Abs Return (%)")
ax.add_artist(legend1)

handles, labels = scatter.legend_elements(prop="sizes", alpha=0.6, num=5)
legend2 = ax.legend(handles, labels, loc="center right", title="Volume")

ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
